# 시크릿 이력 정리와 예방

모든 저장소는 임시 폴더에 만들며 실제 GitHub 계정이나 원격 저장소를 변경하지 않습니다.

## 명령을 실행하기 전에 알아둘 Git 상태

Git은 파일을 세 자리에서 봅니다. **작업 폴더(working tree)** 는 지금 편집하는 파일이 있는 자리입니다. **색인(index)** 은 다음 커밋에 넣을 파일 상태를 골라 두는 자리이며, 스테이징 영역(staging area)이라고도 합니다. **커밋(commit)** 은 색인에 고른 상태를 저장소 이력에 기록한 결과입니다. 따라서 `git add app.py`는 파일을 영구 저장하는 명령이 아니라 현재 `app.py`를 색인에 올리는 명령입니다. 이어서 `git commit -m "메시지"`를 실행해야 그 상태가 커밋으로 기록됩니다.

**HEAD** 는 현재 체크아웃한 커밋을 가리킵니다. 보통 HEAD는 `main` 같은 **브랜치(branch)** 이름을 가리키고, 브랜치 이름은 다시 최신 커밋을 가리킵니다. 새 커밋을 만들면 현재 브랜치가 새 커밋으로 이동합니다. 이 절에서는 브랜치와 태그처럼 커밋을 가리키는 이름을 묶어 참조(reference)라고 부릅니다.

명령은 왼쪽부터 `git` → 하위 명령 → 옵션 → 대상으로 읽습니다. 예를 들어 `git status --short`에서 `status`는 상태를 보고, `--short`는 짧은 형식으로 표시하라는 옵션입니다. 코드의 `git("status", "--short", cwd=repo)`는 터미널에서 같은 명령을 실행하도록 인자를 나눈 Python 표현입니다.


## 준비

예제 문자열은 실제 서비스에서 사용할 수 없는 더미 자격 증명입니다. **시크릿(secret)** 은 API 키, 접근 토큰, 비밀번호처럼 권한을 부여하는 값입니다. 실제 값이 노출되면 Git 이력 정리를 기다리지 말고 발급 서비스에서 먼저 폐기하거나 교체해야 합니다. 이미 복제되거나 로그에 남은 값은 저장소 파일을 지워도 다시 통제할 수 없기 때문입니다.

이 실습은 `DEMO-ONLY-NOT-A-REAL-SECRET`만 사용합니다. 원격도 임시 폴더의 bare 저장소이므로 외부에 게시되지 않습니다. 이후 출력의 “0건”은 이 임시 저장소에서 검사한 참조에 더미 값이 없다는 뜻이며 외부 캐시나 다른 복제본까지 삭제했다는 뜻은 아닙니다.

In [1]:
# 셀이 만든 저장소만 사용하는 임시 작업 폴더를 준비합니다.
import subprocess
import sys
import tempfile
from pathlib import Path
workspace = tempfile.TemporaryDirectory(prefix="git-lesson-")
root = Path(workspace.name)
# cwd는 명령을 실행할 폴더이며 check=True는 명령 실패를 예외로 알립니다.
def git(*args: str, cwd: Path | None = None, check: bool = True) -> str:
    """지정한 임시 저장소에서 Git 명령을 실행합니다."""
    result = subprocess.run(["git", *args], cwd=cwd or root,
                            text=True, capture_output=True, check=check)
    return result.stdout.strip() or result.stderr.strip()
def configure(repo: Path) -> None:
    """실습 저장소에만 작성자 정보를 설정합니다."""
    git("config", "user.name", "Git Learner", cwd=repo)
    git("config", "user.email", "learner@example.invalid", cwd=repo)

In [2]:
# 실제 인증에 사용할 수 없는 더미 문자열만 사용합니다.
DUMMY = "DEMO-ONLY-NOT-A-REAL-SECRET"
remote = root / "secret-remote.git"
git("init", "--bare", "--initial-branch=main", remote.as_posix())
repo = root / "secret-demo"
git("clone", remote.as_posix(), repo.as_posix())
configure(repo)
(repo / "app.py").write_text('print("safe app")\n', encoding="utf-8")
git("add", "app.py", cwd=repo)
git("commit", "-m", "feat: 안전한 앱 시작", cwd=repo)
git("push", "-u", "origin", "main", cwd=repo)
print("더미만 사용:", DUMMY)

더미만 사용: DEMO-ONLY-NOT-A-REAL-SECRET


## 기본 실행

추적 전에는 `.gitignore`와 스테이징한 변경 검사를 함께 적용합니다. `git check-ignore -v .env`에서 `-v`는 어떤 규칙이 어느 파일에 적용됐는지 보여 줍니다. `git ls-files .env`는 색인에 등록된 경로를 찾으며, 빈 출력이면 `.env`가 현재 커밋 후보에 포함되지 않았다는 뜻입니다.

`.gitignore`는 아직 추적하지 않는 경로를 기본 `git add` 대상에서 제외할 뿐 파일을 암호화하거나 과거 커밋을 지우지 않습니다. `git diff --cached -- settings.example`은 다음 커밋에 들어갈 실제 내용을 보여 주므로 커밋 직전 시크릿 검사 지점으로 사용합니다.

In [3]:
# .env를 추적하기 전에 무시 규칙이 적용되는지 확인합니다.
(repo / ".gitignore").write_text(".env\n", encoding="utf-8")
(repo / ".env").write_text(f"DEMO_API_KEY={DUMMY}\n", encoding="utf-8")
git("add", ".gitignore", cwd=repo)
git("commit", "-m", "chore: 환경 파일 제외", cwd=repo)
print(git("check-ignore", "-v", ".env", cwd=repo))
print("색인에 없음:", git("ls-files", ".env", cwd=repo) == "")

.gitignore:1:.env	.env
색인에 없음: True


## 값 하나를 바꿔 검사

안전한 설정 예시는 커밋할 수 있지만 더미 키 패턴이 있으면 검사를 실패로 판정합니다.

In [4]:
# 스테이징한 변경은 이번 커밋에 실제로 포함될 내용만 보여 줍니다.
(repo / "settings.example").write_text("DEMO_API_KEY=replace-me\n", encoding="utf-8")
git("add", "settings.example", cwd=repo)
staged = git("diff", "--cached", "--", "settings.example", cwd=repo)
contains_dummy = DUMMY in staged
print("더미 자격 증명 발견:", contains_dummy)
print(staged)
assert not contains_dummy
git("commit", "-m", "docs: 설정 예시 추가", cwd=repo)
print("안전한 설정 예시 커밋 완료")

더미 자격 증명 발견: False
diff --git a/settings.example b/settings.example
new file mode 100644
index 0000000..02e3d73
--- /dev/null
+++ b/settings.example
@@ -0,0 +1 @@
+DEMO_API_KEY=replace-me
안전한 설정 예시 커밋 완료


## 실패 재현

파일을 다음 커밋에서 삭제해도 이전 커밋 객체에는 내용이 남습니다.

In [5]:
# 가상 키 파일을 다음 커밋에서 삭제해도 과거 커밋에는 값이 남습니다.
git("check-ignore", "-q", ".env", cwd=repo)
git("add", "-f", ".env", cwd=repo)
git("commit", "-m", "test: 더미 키를 잘못 기록", cwd=repo)
leak_commit = git("rev-parse", "HEAD", cwd=repo)
git("rm", ".env", cwd=repo)
git("commit", "-m", "fix: 환경 파일 삭제", cwd=repo)
git("push", cwd=repo)
historic = git("show", f"{leak_commit}:.env", cwd=repo)
print("현재 파일 존재:", (repo / ".env").exists())
print("이전 커밋 내용:", historic.strip())
assert DUMMY in historic

현재 파일 존재: False
이전 커밋 내용: DEMO_API_KEY=DEMO-ONLY-NOT-A-REAL-SECRET


## 수정

격리된 예제 저장소의 모든 참조에서 `.env` 경로를 제거한 뒤 원격을 강제로 갱신합니다. `git filter-repo --force`의 `--force`는 새로 복제한 저장소인지 확인하는 안전 검사를 완화합니다. 이 실습은 방금 만든 임시 mirror에만 사용하며 실제 작업 저장소에서 실행할 이유가 되지 않습니다.

`git rev-list --objects --all`의 각 행은 객체 해시와 경로를 담습니다. `objects.splitlines()`가 행 목록을 만들고, `line.endswith(" .env")`는 공백 뒤 경로가 정확히 `.env`로 끝나는 행인지 검사합니다. `any(...)`는 그런 행이 하나라도 있으면 `True`, 하나도 없으면 `False`를 반환합니다. 따라서 출력 `False`는 검사한 모든 참조에서 `.env` 경로를 찾지 못했다는 뜻입니다. 별도의 `git log -S` 결과 0건은 더미 문자열을 추가하거나 제거한 커밋도 찾지 못했다는 뜻입니다.

`len(value_hits.splitlines()) if value_hits else 0`은 결과 문자열이 있으면 행 수를 세고, 빈 문자열이면 0을 사용합니다. 따라서 검색 결과가 없을 때 출력되는 0은 빈 결과를 숫자로 표시한 값입니다.

In [6]:
# 격리된 원격의 모든 참조를 갱신하고 새 복제본에서 경로와 값을 검사합니다.
clean = root / "clean-history"
git("clone", "--mirror", remote.as_posix(), clean.as_posix())
result = subprocess.run(
    ["git", "filter-repo", "--force", "--path", ".env", "--invert-paths"],
    cwd=clean, text=True, capture_output=True, check=True,
)
git("remote", "add", "origin", remote.as_posix(), cwd=clean)
git("push", "--force", "--mirror", cwd=clean)
objects = git("rev-list", "--objects", "--all", cwd=clean)
value_hits = git("log", "-S", DUMMY, "--all", "--format=%H", cwd=clean)
print("정리 후 .env 경로 있음:", any(line.endswith(" .env") for line in objects.splitlines()))
print("정리 후 더미 값 커밋 수:", len(value_hits.splitlines()) if value_hits else 0)
verify = root / "verify-remote"
git("clone", "--mirror", remote.as_posix(), verify.as_posix())
remote_objects = git("rev-list", "--objects", "--all", cwd=verify)
remote_hits = git("log", "-S", DUMMY, "--all", "--format=%H", cwd=verify)
print("원격 재검사 값 커밋 수:", len(remote_hits.splitlines()) if remote_hits else 0)
assert not any(line.endswith(" .env") for line in objects.splitlines())
assert not any(line.endswith(" .env") for line in remote_objects.splitlines())
assert not value_hits and not remote_hits

정리 후 .env 경로 있음: False
정리 후 더미 값 커밋 수: 0
원격 재검사 값 커밋 수: 0
